In [1]:
import pandas as pd
import numpy as np

ledger = pd.read_csv("ledger.csv")
gateway = pd.read_csv("gateway.csv")

In [2]:
print("Ledger Nulls:\n", ledger.isnull().sum())
print("Gateway Nulls:\n", gateway.isnull().sum())

print("Ledger Duplicates:", ledger.duplicated().sum())
print("Gateway Duplicates:", gateway.duplicated().sum())

Ledger Nulls:
 transaction_id      0
transaction_date    0
merchant_id         0
amount_usd          0
status              0
payment_method      0
dtype: int64
Gateway Nulls:
 transaction_id      0
transaction_date    0
merchant_id         0
amount_usd          0
status              0
payment_method      0
dtype: int64
Ledger Duplicates: 0
Gateway Duplicates: 0


In [5]:
key = "transaction_id"
missing_in_gateway = ledger[~ledger[key].isin(gateway[key])]
missing_in_gateway.to_csv("missing_in_gateway.csv", index=False)
missing_in_ledger = gateway[~gateway[key].isin(ledger[key])]
missing_in_ledger.to_csv("missing_in_ledger.csv", index=False)

In [6]:
merged = pd.merge(
    ledger,
    gateway,
    on=key,
    how="inner",
    suffixes=("_ledger", "_gateway")
)

In [8]:
amount_mismatches = merged[
    merged["amount_usd_ledger"] != merged["amount_usd_gateway"]
]

amount_mismatches.to_csv("amount_mismatches.csv", index=False)

In [9]:
status_mismatches = merged[
    merged["status_ledger"] != merged["status_gateway"]
]

status_mismatches.to_csv("status_mismatches.csv", index=False)

In [11]:
merged["amount_match"] = merged["amount_usd_ledger"] == merged["amount_usd_gateway"]
merged["status_match"] = merged["status_ledger"] == merged["status_gateway"]

merged["reconciliation_status"] = np.where(
    (merged["amount_match"]) & (merged["status_match"]),
    "matched",
    "mismatched"
)

merged.to_csv("reconciliation_report.csv", index=False)

In [12]:
summary = {
    "total_ledger_records": len(ledger),
    "total_gateway_records": len(gateway),
    "missing_in_gateway": len(missing_in_gateway),
    "missing_in_ledger": len(missing_in_ledger),
    "amount_mismatches": len(amount_mismatches),
    "status_mismatches": len(status_mismatches),
    "fully_matched": len(merged[
        (merged["amount_match"]) & (merged["status_match"])
    ])
}

import json
with open("summary_metrics.json", "w") as f:
    json.dump(summary, f, indent=4)

print(summary)

{'total_ledger_records': 10, 'total_gateway_records': 9, 'missing_in_gateway': 2, 'missing_in_ledger': 1, 'amount_mismatches': 2, 'status_mismatches': 1, 'fully_matched': 5}
